In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# GA Configuration
config = {
    # Genetic Algorithm Parameters
    'population_size': 100,
    'num_generations': 200,
    'mutation_rate': 0.1,
    'crossover_rate': 0.8,
    'elite_size': 5,
    'tournament_size': 5,

    # Delivery Time Constraints (in minutes)
    'min_delivery_time': 6,
    'max_delivery_time': 20,

    # Service radius (in km, converted to delivery time)
    'service_radius_km': 5.0,
    'avg_speed_kmh': 30.0,  # Average delivery speed in km/h

    # Fitness Weights
    'weight_population_coverage': 0.3,
    'weight_accessibility': 0.2,
    'weight_competition': -0.15,
    'weight_road_connectivity': 0.15,
    'weight_poi_proximity': 0.1,
    'weight_facility_count': -0.1,  # Penalize too many facilities

    # Constraint Penalties
    'penalty_delivery_time_violation': 1000,
    'penalty_overlap': 500,

    # Candidate Generation
    'num_candidates': 200,
    'grid_size': 0.01,  # Grid spacing in degrees

    # Stopping Criteria
    'convergence_patience': 20,
    'min_improvement': 0.001
}

print("Configuration loaded successfully!")
print(f"Population Size: {config['population_size']}")
print(f"Number of Generations: {config['num_generations']}")
print(f"Delivery Time Range: {config['min_delivery_time']}-{config['max_delivery_time']} minutes")

Configuration loaded successfully!
Population Size: 100
Number of Generations: 200
Delivery Time Range: 6-20 minutes


In [ ]:
# Load all datasets
data_path = r"/content"

# Load existing locations (for competition analysis)
existing_locations = pd.read_csv(f"{data_path}/DarkStore_Final_Result.csv")
print(f"Loaded existing locations: {existing_locations.shape[0]} records")

# Load POIs
pois = pd.read_csv(f"{data_path}/nasr_city_pois_cleaned.csv")
print(f"Loaded POIs: {pois.shape[0]} records")

# Load population points
population_points = pd.read_csv(f"{data_path}/nasr_city_population_points_cleaned_v2.csv")
print(f"Loaded population points: {population_points.shape[0]} records")

# Load roads
roads = pd.read_csv(f"{data_path}/nasr_city_roads_with_coordinates.csv")

# Filter roads to keep only Cairo/Nasr City coordinates (Egypt: lat ~30, lon ~31)
valid_lat_range = (25.0, 32.0)
valid_lon_range = (25.0, 35.0)
roads = roads[
    (roads['latitude'] >= valid_lat_range[0]) &
    (roads['latitude'] <= valid_lat_range[1]) &
    (roads['longitude'] >= valid_lon_range[0]) &
    (roads['longitude'] <= valid_lon_range[1])
]
print(f"Loaded road segments: {roads.shape[0]} records (filtered from original)")

# Load features (for reference)
features_df = pd.read_csv(f"{data_path}/features_extracted.csv")
print(f"Loaded features: {features_df.shape[0]} records")

print("\nAll datasets loaded successfully!")

Loaded existing locations: 215 records
Loaded POIs: 1040 records
Loaded population points: 8103 records
Loaded road segments: 322612 records (filtered from original)
Loaded features: 215 records

All datasets loaded successfully!


In [ ]:
# Display basic statistics
print("=== Existing Locations Summary ===")
print(existing_locations[['latitude', 'longitude', 'Population_Density', 'Competitor_Density', 'Final_Score']].describe())

print("\n=== Population Points Summary ===")
print(population_points.describe())

print("\n=== POIs Summary ===")
print(pois['category'].value_counts().head(10))

print("\n=== Roads Summary ===")
print(roads['road_type'].value_counts())

=== Existing Locations Summary ===
         latitude   longitude  Population_Density  Competitor_Density  \
count  215.000000  215.000000        2.150000e+02        2.150000e+02   
mean    29.874964   30.816210        1.652425e-16        3.304850e-17   
std      1.075910    8.689005        1.002334e+00        1.002334e+00   
min     21.415390  -95.567638       -1.830624e+00       -1.707963e+00   
25%     30.039593   31.334474       -3.030272e-01       -8.603642e-01   
50%     30.049064   31.345291        2.513558e-01        1.083199e-01   
75%     30.066269   31.392976        6.245756e-01        7.944712e-01   
max     31.342846   39.886981        2.293311e+00        1.440261e+00   

       Final_Score  
count   215.000000  
mean      0.728387  
std       0.107367  
min       0.000000  
25%       0.717417  
50%       0.756087  
75%       0.777045  
max       1.000000  

=== Population Points Summary ===
          latitude    longitude   population
count  8103.000000  8103.000000  8103.

In [ ]:
# Determine study area bounds from data
# Use only existing_locations, population_points, and pois for bounds
# DO NOT filter - use actual data to determine the true study area

print("Calculating study area bounds from actual data...")

# Calculate bounds from each dataset separately
existing_bounds = {
    'min_lat': existing_locations['latitude'].min(),
    'max_lat': existing_locations['latitude'].max(),
    'min_lon': existing_locations['longitude'].min(),
    'max_lon': existing_locations['longitude'].max()
}

population_bounds = {
    'min_lat': population_points['latitude'].min(),
    'max_lat': population_points['latitude'].max(),
    'min_lon': population_points['longitude'].min(),
    'max_lon': population_points['longitude'].max()
}

poi_bounds = {
    'min_lat': pois['latitude'].min(),
    'max_lat': pois['latitude'].max(),
    'min_lon': pois['longitude'].min(),
    'max_lon': pois['longitude'].max()
}

print("Existing locations bounds:", existing_bounds)
print("Population points bounds:", population_bounds)
print("POI bounds:", poi_bounds)

# Use the intersection of all bounds to ensure all data overlaps
bounds = {
    'min_lat': max(existing_bounds['min_lat'], population_bounds['min_lat'], poi_bounds['min_lat']),
    'max_lat': min(existing_bounds['max_lat'], population_bounds['max_lat'], poi_bounds['max_lat']),
    'min_lon': max(existing_bounds['min_lon'], population_bounds['min_lon'], poi_bounds['min_lon']),
    'max_lon': min(existing_bounds['max_lon'], population_bounds['max_lon'], poi_bounds['max_lon'])
}

print("\nFinal Study Area Bounds (intersection):")
print(f"Latitude: {bounds['min_lat']:.4f} to {bounds['max_lat']:.4f}")
print(f"Longitude: {bounds['min_lon']:.4f} to {bounds['max_lon']:.4f}")

# Verify bounds are reasonable for Nasr City, Egypt
if bounds['min_lat'] < 29 or bounds['max_lat'] > 31.5 or bounds['min_lon'] < 30 or bounds['max_lon'] > 32:
    print("\nWARNING: Bounds seem outside expected Nasr City range")
    print("Using population points as primary reference instead...")
    bounds = population_bounds.copy()
    print(f"Adjusted bounds - Lat: {bounds['min_lat']:.4f} to {bounds['max_lat']:.4f}, Lon: {bounds['min_lon']:.4f} to {bounds['max_lon']:.4f}")

Calculating study area bounds from actual data...
Existing locations bounds: {'min_lat': 21.41539, 'max_lat': 31.3428457, 'min_lon': -95.5676383, 'max_lon': 39.8869806}
Population points bounds: {'min_lat': 29.99125022, 'max_lat': 30.08625022, 'min_lon': 31.31374915, 'max_lon': 31.43374915}
POI bounds: {'min_lat': 30.0150124, 'max_lat': 30.1301796, 'min_lon': 31.3008353, 'max_lon': 31.4198317}

Final Study Area Bounds (intersection):
Latitude: 30.0150 to 30.0863
Longitude: 31.3137 to 31.4198


#  Generate Candidate Locations

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    # Radius of earth in kilometers
    r = 6371

    return c * r

def generate_candidate_locations(bounds, num_candidates, method='grid'):
    """
    Generate candidate dark store locations within the study area.

    Parameters:
    -----------
    bounds : dict
        Dictionary with min_lat, max_lat, min_lon, max_lon
    num_candidates : int
        Number of candidate locations to generate
    method : str
        'grid' for grid-based generation, 'random' for random generation

    Returns:
    --------
    candidates : DataFrame
        DataFrame with candidate location coordinates
    """
    if method == 'grid':
        # Generate grid-based candidates
        lat_range = np.linspace(bounds['min_lat'], bounds['max_lat'],
                                 int(np.sqrt(num_candidates)))
        lon_range = np.linspace(bounds['min_lon'], bounds['max_lon'],
                                 int(np.sqrt(num_candidates)))

        candidates = []
        for lat in lat_range:
            for lon in lon_range:
                candidates.append({'latitude': lat, 'longitude': lon})

        candidates_df = pd.DataFrame(candidates)
        # If we have too many, sample down
        if len(candidates_df) > num_candidates:
            candidates_df = candidates_df.sample(n=num_candidates, random_state=RANDOM_SEED)

    else:  # random
        lats = np.random.uniform(bounds['min_lat'], bounds['max_lat'], num_candidates)
        lons = np.random.uniform(bounds['min_lon'], bounds['max_lon'], num_candidates)
        candidates_df = pd.DataFrame({'latitude': lats, 'longitude': lons})

    # Add candidate ID
    candidates_df['candidate_id'] = range(len(candidates_df))

    return candidates_df.reset_index(drop=True)

# Generate candidate locations
candidates = generate_candidate_locations(bounds, config['num_candidates'], method='grid')
print(f"Generated {len(candidates)} candidate locations")
print(candidates.head())

# DIAGNOSTIC: Check spatial overlap between candidates and population points
print("\n=== SPATIAL OVERLAP DIAGNOSTIC ===")
print(f"Candidate bounds: Lat [{bounds['min_lat']:.4f}, {bounds['max_lat']:.4f}], Lon [{bounds['min_lon']:.4f}, {bounds['max_lon']:.4f}]")
print(f"Population bounds: Lat [{population_points['latitude'].min():.4f}, {population_points['latitude'].max():.4f}], Lon [{population_points['longitude'].min():.4f}, {population_points['longitude'].max():.4f}]")

# Calculate delivery time distance bounds
min_dist_km = (config['min_delivery_time'] / 60) * config['avg_speed_kmh']
max_dist_km = (config['max_delivery_time'] / 60) * config['avg_speed_kmh']
print(f"\nDelivery time {config['min_delivery_time']}-{config['max_delivery_time']} min = {min_dist_km:.2f}-{max_dist_km:.2f} km")

# Check if candidates are within delivery time range of population points
sample_candidate = candidates.iloc[0]
sample_distances = population_points.apply(
    lambda row: haversine_distance(
        sample_candidate['latitude'], sample_candidate['longitude'],
        row['latitude'], row['longitude']
    ), axis=1
)
print(f"\nSample candidate distances to population: min={sample_distances.min():.2f}km, max={sample_distances.max():.2f}km")
# Fix: Correctly sum the boolean condition to get the count of points within range
points_within_range_count = ((sample_distances >= min_dist_km) & (sample_distances <= max_dist_km)).sum()
print(f"Population points within delivery range: {points_within_range_count}/{len(sample_distances)}")

# If overlap is poor, use population bounds instead
if points_within_range_count < len(sample_distances) * 0.1:
    print("\nWARNING: Poor spatial overlap detected!")
    print("Using population point bounds for candidate generation instead...")
    bounds = {
        'min_lat': population_points['latitude'].min(),
        'max_lat': population_points['latitude'].max(),
        'min_lon': population_points['longitude'].min(),
        'max_lon': population_points['longitude'].max()
    }
    candidates = generate_candidate_locations(bounds, config['num_candidates'], method='grid')
    print(f"Regenerated {len(candidates)} candidates using population bounds")
    print(f"New bounds: Lat [{bounds['min_lat']:.4f}, {bounds['max_lat']:.4f}], Lon [{bounds['min_lon']:.4f}, {bounds['max_lon']:.4f}]")

Generated 196 candidate locations
    latitude  longitude  candidate_id
0  30.015012  31.313749             0
1  30.015012  31.321909             1
2  30.015012  31.330070             2
3  30.015012  31.338230             3
4  30.015012  31.346390             4

=== SPATIAL OVERLAP DIAGNOSTIC ===
Candidate bounds: Lat [30.0150, 30.0863], Lon [31.3137, 31.4198]
Population bounds: Lat [29.9913, 30.0863], Lon [31.3137, 31.4337]

Delivery time 6-20 min = 3.00-10.00 km

Sample candidate distances to population: min=2.43km, max=13.55km
Population points within delivery range: 7088/8103


##  Feature Engineering for Candidate Locations

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    # Radius of earth in kilometers
    r = 6371

    return c * r

def calculate_distance_to_nearest(candidate_lat, candidate_lon,
                                   reference_df, lat_col='latitude', lon_col='longitude'):
    """
    Calculate distance from candidate to nearest point in reference dataframe.
    """
    distances = reference_df.apply(
        lambda row: haversine_distance(
            candidate_lat, candidate_lon,
            row[lat_col], row[lon_col]
        ), axis=1
    )
    return distances.min()

def calculate_distance_to_k_nearest(candidate_lat, candidate_lon,
                                     reference_df, k=5, lat_col='latitude', lon_col='longitude'):
    """
    Calculate distances to k nearest points.
    """
    distances = reference_df.apply(
        lambda row: haversine_distance(
            candidate_lat, candidate_lon,
            row[lat_col], row[lon_col]
        ), axis=1
    )
    return distances.nsmallest(k).values

def calculate_population_density_around(candidate_lat, candidate_lon,
                                         population_df, radius_km=2):
    """
    Calculate total population within radius of candidate location.
    """
    distances = population_df.apply(
        lambda row: haversine_distance(
            candidate_lat, candidate_lon,
            row['latitude'], row['longitude']
        ), axis=1
    )

    # Filter points within radius
    within_radius = population_df[distances <= radius_km]

    if len(within_radius) == 0:
        return 0

    return within_radius['population'].sum()

print("Distance calculation functions defined!")

Distance calculation functions defined!


In [ ]:
def engineer_features_for_candidates(candidates, existing_locations, pois,
                                      population_points, roads, config):
    """
    Engineer features for all candidate locations.
    """
    print("Engineering features for candidate locations...")

    features = candidates.copy()

    # Initialize feature columns
    features['population_coverage'] = 0
    features['distance_to_nearest_road'] = 0
    features['distance_to_nearest_poi'] = 0
    features['poi_count_within_2km'] = 0
    features['distance_to_nearest_competitor'] = 0
    features['competitor_density'] = 0

    # Calculate features for each candidate
    for idx, row in candidates.iterrows():
        lat, lon = row['latitude'], row['longitude']

        # Population coverage (within 2km)
        features.loc[idx, 'population_coverage'] = calculate_population_density_around(
            lat, lon, population_points, radius_km=2
        )

        # Distance to nearest road
        features.loc[idx, 'distance_to_nearest_road'] = calculate_distance_to_nearest(
            lat, lon, roads
        )

        # Distance to nearest POI
        features.loc[idx, 'distance_to_nearest_poi'] = calculate_distance_to_nearest(
            lat, lon, pois
        )

        # POI count within 2km
        poi_distances = pois.apply(
            lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']),
            axis=1
        )
        features.loc[idx, 'poi_count_within_2km'] = (poi_distances <= 2).sum()

        # Distance to nearest competitor (existing supermarket/warehouse)
        competitors = existing_locations[
            existing_locations['category'].isin(['supermarket', 'warehouse', 'grocery store'])
        ]
        if len(competitors) > 0:
            features.loc[idx, 'distance_to_nearest_competitor'] = calculate_distance_to_nearest(
                lat, lon, competitors
            )

            # Competitor density (within 2km)
            competitor_distances = competitors.apply(
                lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']),
                axis=1
            )
            features.loc[idx, 'competitor_density'] = (competitor_distances <= 2).sum()

        # Progress indicator
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(candidates)} candidates...")

    # Calculate derived features
    features['accessibility_score'] = 1 / (features['distance_to_nearest_road'] + 0.01)
    features['poi_proximity_score'] = 1 / (features['distance_to_nearest_poi'] + 0.01)
    features['competition_penalty'] = features['competitor_density'] / (features['distance_to_nearest_competitor'] + 0.01)

    print("Feature engineering completed!")
    return features

# Engineer features
candidates_with_features = engineer_features_for_candidates(
    candidates, existing_locations, pois, population_points, roads, config
)

print(f"\nFeature statistics:")
print(candidates_with_features.describe())

Engineering features for candidate locations...
Processed 50/196 candidates...
Processed 100/196 candidates...
Processed 150/196 candidates...
Feature engineering completed!

Feature statistics:
         latitude   longitude  candidate_id  population_coverage  \
count  196.000000  196.000000     196.00000           196.000000   
mean    30.050631   31.366790      97.50000         96719.003457   
std      0.022146    0.032979      56.72448         77785.490811   
min     30.015012   31.313749       0.00000             0.000000   
25%     30.031452   31.338230      48.75000         26318.215492   
50%     30.050631   31.366790      97.50000         81334.042650   
75%     30.069811   31.395351     146.25000        159624.019314   
max     30.086250   31.419832     195.00000        269580.210673   

       distance_to_nearest_road  distance_to_nearest_poi  \
count                196.000000               196.000000   
mean                   0.142681                 0.561861   
std         

In [ ]:
def normalize_features(df, feature_columns):
    """
    Normalize features using Min-Max scaling.
    """
    scaler = MinMaxScaler()
    normalized = df.copy()

    for col in feature_columns:
        if col in df.columns:
            normalized[col] = scaler.fit_transform(df[[col]])

    return normalized

# Features to normalize
feature_columns = [
    'population_coverage',
    'distance_to_nearest_road',
    'distance_to_nearest_poi',
    'poi_count_within_2km',
    'distance_to_nearest_competitor',
    'competitor_density',
    'accessibility_score',
    'poi_proximity_score',
    'competition_penalty'
]

# Use the already-engineered features from cell 14, don't re-run feature engineering
candidates_normalized = normalize_features(candidates_with_features, feature_columns)

print("Features normalized successfully!")
print(candidates_normalized[feature_columns].describe())

Features normalized successfully!
       population_coverage  distance_to_nearest_road  distance_to_nearest_poi  \
count           196.000000                196.000000               196.000000   
mean              0.358776                  0.156260                 0.264020   
std               0.288543                  0.188130                 0.231050   
min               0.000000                  0.000000                 0.000000   
25%               0.097627                  0.031039                 0.082993   
50%               0.301706                  0.075512                 0.193541   
75%               0.592121                  0.219185                 0.370781   
max               1.000000                  1.000000                 1.000000   

       poi_count_within_2km  distance_to_nearest_competitor  \
count            196.000000                      196.000000   
mean               0.248498                        0.266814   
std                0.258450                    

In [ ]:
class GeneticAlgorithmOptimizer:
    """
    Genetic Algorithm for Dark Store Location Optimization.
    Optimized with precomputed distances and vectorized operations.
    """

    def __init__(self, candidates, config, population_points):
        self.candidates = candidates
        self.config = config
        self.population_points = population_points
        self.num_candidates = len(candidates)
        self.fitness_history = []
        self.best_chromosome = None
        self.best_fitness = -float('inf')

        # Precompute all distances for performance
        print("Precomputing distance matrices for performance optimization...")
        self.precompute_all_distances()

    def precompute_all_distances(self):
        """Precompute all distance matrices using vectorized operations."""
        n_candidates = self.num_candidates
        n_pop = len(self.population_points)

        # Extract coordinates as NumPy arrays for vectorization
        candidate_coords = self.candidates[['latitude', 'longitude']].values
        pop_coords = self.population_points[['latitude', 'longitude']].values
        pop_values = self.population_points['population'].values

        print(f"Computing distance matrix: {n_candidates} candidates × {n_pop} population points")

        # Convert all coordinates to radians at once
        candidate_lats = np.radians(candidate_coords[:, 0])
        candidate_lons = np.radians(candidate_coords[:, 1])
        pop_lats = np.radians(pop_coords[:, 0])
        pop_lons = np.radians(pop_coords[:, 1])

        # Precompute distance matrix (candidates × population_points)
        # Using broadcasting for vectorized computation
        self.distance_matrix = np.zeros((n_candidates, n_pop))

        for i in range(n_candidates):
            lat1 = candidate_lats[i]
            lon1 = candidate_lons[i]

            dlat = pop_lats - lat1
            dlon = pop_lons - lon1

            a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(pop_lats) * np.sin(dlon/2)**2
            c = 2 * np.arcsin(np.sqrt(a))

            self.distance_matrix[i] = c * 6371  # Earth radius in km

        # Precompute candidate-to-candidate distance matrix for overlap calculation
        print("Computing candidate-to-candidate distance matrix...")
        self.candidate_distance_matrix = np.zeros((n_candidates, n_candidates))

        for i in range(n_candidates):
            lat1 = candidate_lats[i]
            lon1 = candidate_lons[i]

            dlat = candidate_lats - lat1
            dlon = candidate_lons - lon1

            a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(candidate_lats) * np.sin(dlon/2)**2
            c = 2 * np.arcsin(np.sqrt(a))

            self.candidate_distance_matrix[i] = c * 6371

        # Store population values for quick access
        self.population_values = pop_values

        # Precompute delivery time distance bounds
        self.min_distance_km = (self.config['min_delivery_time'] / 60) * self.config['avg_speed_kmh']
        self.max_distance_km = (self.config['max_delivery_time'] / 60) * self.config['avg_speed_kmh']
        self.overlap_threshold = self.max_distance_km * 0.5

        print("Distance matrices precomputed!")
        print(f"Distance matrix shape: {self.distance_matrix.shape}")
        print(f"Sample distances from candidate 0: min={self.distance_matrix[0].min():.2f}km, max={self.distance_matrix[0].max():.2f}km")
        print(f"Delivery time bounds: {self.min_distance_km:.2f}km - {self.max_distance_km:.2f}km")

    def initialize_population(self):
        """
        Initialize population with random binary chromosomes.
        Each chromosome represents which candidate locations are selected (1) or not (0).
        """
        population = []
        for _ in range(self.config['population_size']):
            # Random chromosome with 10-30% of locations selected
            num_selected = np.random.randint(
                max(1, int(self.num_candidates * 0.1)),
                int(self.num_candidates * 0.3) + 1
            )
            chromosome = np.zeros(self.num_candidates, dtype=int)
            selected_indices = np.random.choice(self.num_candidates, num_selected, replace=False)
            chromosome[selected_indices] = 1
            population.append(chromosome)

        # Verify population diversity
        unique_counts = set(np.sum(chrom) for chrom in population)
        print(f"Population initialized with {len(unique_counts)} different selection counts: {sorted(unique_counts)}")

        return np.array(population)

    def calculate_population_coverage_for_solution(self, chromosome):
        """
        Calculate total population coverage for a given solution using precomputed distances.
        Vectorized implementation for performance.
        """
        selected_indices = np.where(chromosome == 1)[0]

        if len(selected_indices) == 0:
            return 0

        total_coverage = 0
        covered_mask = np.zeros(len(self.population_values), dtype=bool)

        for idx in selected_indices:
            # Get precomputed distances for this candidate
            distances = self.distance_matrix[idx]

            # Points within acceptable delivery time range (vectorized)
            within_range = (distances >= self.min_distance_km) & (distances <= self.max_distance_km)

            # Update covered mask
            covered_mask = covered_mask | within_range

        # Sum population of covered points
        total_coverage = np.sum(self.population_values[covered_mask])

        return total_coverage

    def calculate_overlap_penalty(self, chromosome):
        """
        Calculate penalty for overlapping coverage areas using precomputed distances.
        """
        selected_indices = np.where(chromosome == 1)[0]

        if len(selected_indices) < 2:
            return 0

        penalty = 0

        # Use precomputed candidate-to-candidate distances
        for i in range(len(selected_indices)):
            for j in range(i + 1, len(selected_indices)):
                idx1, idx2 = selected_indices[i], selected_indices[j]

                distance = self.candidate_distance_matrix[idx1, idx2]

                # If coverage areas overlap significantly
                if distance < self.overlap_threshold:
                    penalty += self.config['penalty_overlap']

        return penalty

    def fitness_function(self, chromosome):
        """
        Calculate fitness for a chromosome.
        Higher fitness is better.
        """
        selected_indices = np.where(chromosome == 1)[0]

        if len(selected_indices) == 0:
            return -float('inf')

        # Get features for selected candidates (using NumPy for speed)
        selected_candidates = self.candidates.iloc[selected_indices]

        # Calculate components using NumPy operations
        population_coverage = self.calculate_population_coverage_for_solution(chromosome)

        avg_accessibility = selected_candidates['accessibility_score'].mean()
        avg_poi_proximity = selected_candidates['poi_proximity_score'].mean()
        avg_competition = selected_candidates['competition_penalty'].mean()

        # Calculate overlap penalty
        overlap_penalty = self.calculate_overlap_penalty(chromosome)

        # Calculate facility count penalty
        facility_count_penalty = len(selected_indices) * self.config['weight_facility_count'] * 100

        # Calculate weighted fitness
        fitness = (
            self.config['weight_population_coverage'] * population_coverage / 1000 +
            self.config['weight_accessibility'] * avg_accessibility +
            self.config['weight_poi_proximity'] * avg_poi_proximity +
            self.config['weight_competition'] * avg_competition +
            facility_count_penalty -
            overlap_penalty
        )

        return fitness

    def tournament_selection(self, population, fitness_scores, tournament_size):
        """
        Select parent using tournament selection.
        """
        tournament_indices = np.random.choice(len(population), tournament_size, replace=False)
        tournament_fitness = [fitness_scores[i] for i in tournament_indices]
        winner_index = tournament_indices[np.argmax(tournament_fitness)]
        return population[winner_index]

    def crossover(self, parent1, parent2):
        """
        Perform uniform crossover between two parents.
        """
        if np.random.random() > self.config['crossover_rate']:
            return parent1.copy(), parent2.copy()

        # Uniform crossover (vectorized)
        mask = np.random.randint(0, 2, size=len(parent1), dtype=int)
        child1 = np.where(mask, parent1, parent2)
        child2 = np.where(mask, parent2, parent1)

        return child1, child2

    def mutate(self, chromosome):
        """
        Perform bit-flip mutation (vectorized where possible).
        """
        # Generate random values for all positions
        random_values = np.random.random(len(chromosome))
        # Flip bits where random value < mutation rate
        mutation_mask = random_values < self.config['mutation_rate']
        chromosome[mutation_mask] = 1 - chromosome[mutation_mask]

        # Ensure at least one location is selected
        if np.sum(chromosome) == 0:
            chromosome[np.random.randint(0, len(chromosome))] = 1

        return chromosome

    def evolve(self):
        """
        Run the genetic algorithm optimization with improved progress reporting.
        """
        print("Starting Genetic Algorithm optimization...")
        print(f"Population size: {self.config['population_size']}")
        print(f"Max generations: {self.config['num_generations']}")
        print(f"Convergence patience: {self.config['convergence_patience']}")
        print("-" * 50)

        # Initialize population
        population = self.initialize_population()

        # Test initial fitness diversity
        initial_fitness = [self.fitness_function(chrom) for chrom in population[:5]]
        print(f"Sample initial fitness values: {[f'{f:.2f}' for f in initial_fitness]}")

        # Track best solution
        patience_counter = 0
        previous_best_fitness = -float('inf')
        start_time = pd.Timestamp.now()

        for generation in range(self.config['num_generations']):
            generation_start = pd.Timestamp.now()

            # Calculate fitness for all chromosomes
            fitness_scores = [self.fitness_function(chrom) for chrom in population]

            # Check fitness diversity
            unique_fitness = len(set(f"{f:.4f}" for f in fitness_scores))
            if unique_fitness < 5 and generation < 5:
                print(f"Warning: Low fitness diversity ({unique_fitness} unique values) in generation {generation+1}")

            # Track best fitness
            current_best_fitness = max(fitness_scores)
            current_best_idx = np.argmax(fitness_scores)

            if current_best_fitness > self.best_fitness:
                self.best_fitness =  current_best_fitness
                self.best_chromosome = population[current_best_idx].copy()

            self.fitness_history.append(current_best_fitness)

            # Check for convergence
            if current_best_fitness - previous_best_fitness < self.config['min_improvement']:
                patience_counter += 1
            else:
                patience_counter = 0

            previous_best_fitness = current_best_fitness

            # Calculate generation time
            generation_time = (pd.Timestamp.now() - generation_start).total_seconds()

            # Print progress every generation with timing
            print(f"Gen {generation + 1:3d}/{self.config['num_generations']} | "
                  f"Best Fit: {current_best_fitness:8.4f} | "
                  f"Selected: {np.sum(population[current_best_idx]):2d} | "
                  f"Time: {generation_time:.2f}s | "
                  f"Patience: {patience_counter}/{self.config['convergence_patience']}")

            if patience_counter >= self.config['convergence_patience']:
                print(f"\nConverged at generation {generation}")
                break

            # Selection
            new_population = []

            # Elitism: keep best chromosomes
            elite_indices = np.argsort(fitness_scores)[-self.config['elite_size']:]
            for idx in elite_indices:
                new_population.append(population[idx].copy())

            # Generate offspring
            while len(new_population) < self.config['population_size']:
                parent1 = self.tournament_selection(population, fitness_scores,
                                                   self.config['tournament_size'])
                parent2 = self.tournament_selection(population, fitness_scores,
                                                   self.config['tournament_size'])

                child1, child2 = self.crossover(parent1, parent2)
                child1 = self.mutate(child1)
                child2 = self.mutate(child2)

                new_population.append(child1)
                if len(new_population) < self.config['population_size']:
                    new_population.append(child2)

            population = np.array(new_population)

        total_time = (pd.Timestamp.now() - start_time).total_seconds()

        print("-" * 50)
        print(f"\nOptimization completed!")
        print(f"Total runtime: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Best Fitness: {self.best_fitness:.4f}")
        print(f"Number of Selected Locations: {np.sum(self.best_chromosome)}")

        return self.best_chromosome, self.fitness_history

print("Genetic Algorithm class defined successfully!")

Genetic Algorithm class defined successfully!


In [ ]:
# Initialize and run GA
ga = GeneticAlgorithmOptimizer(candidates_normalized, config, population_points)

# Run optimization
best_chromosome, fitness_history = ga.evolve()

print("\n" + "="*50)
print("Optimization Results:")
print("="*50)
print(f"Total Generations: {len(fitness_history)}")
print(f"Final Best Fitness: {fitness_history[-1]:.4f}")
print(f"Number of Selected Dark Stores: {np.sum(best_chromosome)}")

Precomputing distance matrices for performance optimization...
Computing distance matrix: 196 candidates × 8103 population points
Computing candidate-to-candidate distance matrix...
Distance matrices precomputed!
Distance matrix shape: (196, 8103)
Sample distances from candidate 0: min=2.43km, max=13.55km
Delivery time bounds: 3.00km - 10.00km
Starting Genetic Algorithm optimization...
Population size: 100
Max generations: 200
Convergence patience: 20
--------------------------------------------------
Population initialized with 35 different selection counts: [np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(46), np.int64(47), np.int64(48), np.int64(50), np.int64(52), np.int64(53), np.int64(54),

In [ ]:
# Extract selected locations
selected_indices = np.where(best_chromosome == 1)[0]
selected_locations = candidates_normalized.iloc[selected_indices].copy()

# Calculate delivery time statistics for each selected location
def calculate_delivery_time_stats(location, population_points, config):
    """
    Calculate delivery time statistics for a location.
    Only includes population points within the service radius (15-40 min delivery time).
    """
    # Calculate distances to all population points
    distances = population_points.apply(
        lambda row: haversine_distance(
            location['latitude'], location['longitude'],
            row['latitude'], row['longitude']
        ), axis=1
    )

    # Convert to delivery time
    delivery_times = (distances / config['avg_speed_kmh']) * 60

    # Filter to only include points within acceptable delivery time range
    min_delivery_time = config['min_delivery_time']
    max_delivery_time = config['max_delivery_time']

    valid_mask = (delivery_times >= min_delivery_time) & (delivery_times <= max_delivery_time)
    valid_delivery_times = delivery_times[valid_mask]

    if len(valid_delivery_times) == 0:
        # If no points within range, return statistics for all points (with warning)
        print(f"Warning: No population points within {min_delivery_time}-{max_delivery_time} min range for location ({location['latitude']:.4f}, {location['longitude']:.4f})")
        valid_delivery_times = delivery_times

    return {
        'min_delivery_time': valid_delivery_times.min(),
        'max_delivery_time': valid_delivery_times.max(),
        'mean_delivery_time': valid_delivery_times.mean(),
        'median_delivery_time': valid_delivery_times.median(),
        'std_delivery_time': valid_delivery_times.std(),
        'covered_points': len(valid_delivery_times)
    }

# Add delivery time statistics
delivery_stats = []
for idx, location in selected_locations.iterrows():
    stats = calculate_delivery_time_stats(location, population_points, config)
    delivery_stats.append(stats)

delivery_stats_df = pd.DataFrame(delivery_stats)
selected_locations = pd.concat([selected_locations.reset_index(drop=True), delivery_stats_df], axis=1)

# Add location rank
selected_locations['location_rank'] = range(1, len(selected_locations) + 1)

# Reorder columns
result_columns = [
    'location_rank', 'latitude', 'longitude',
    'population_coverage', 'accessibility_score', 'poi_proximity_score',
    'competition_penalty', 'covered_points',
    'min_delivery_time', 'max_delivery_time',
    'mean_delivery_time', 'median_delivery_time'
]

selected_locations = selected_locations[result_columns]

print("Selected Dark Store Locations:")
print(selected_locations.to_string(index=False))

print(f"\nSummary:")
print(f"Total number of dark stores to open: {len(selected_locations)}")
print(f"Average delivery time: {selected_locations['mean_delivery_time'].mean():.2f} minutes")
print(f"Average population coverage per location: {selected_locations['population_coverage'].mean():.2f}")
print(f"Average covered points per location: {selected_locations['covered_points'].mean():.0f}")

Selected Dark Store Locations:
 location_rank  latitude  longitude  population_coverage  accessibility_score  poi_proximity_score  competition_penalty  covered_points  min_delivery_time  max_delivery_time  mean_delivery_time  median_delivery_time
             1 30.015012  31.313749             0.000000             0.222984             0.009079             0.000824            7088           6.000908          19.997920           13.186090             13.411664
             2 30.015012  31.346390             0.149552             0.172746             0.018999             0.001387            6426           6.004491          19.986122           11.673685             11.569891
             3 30.020492  31.395351             0.012399             0.028639             0.024461             0.000000            6566           6.003581          19.996636           11.343672             11.051570
             4 30.031452  31.313749             0.009459             0.000000             0.003604       

In [ ]:
# Create interactive map
center_lat = (bounds['min_lat'] + bounds['max_lat']) / 2
center_lon = (bounds['min_lon'] + bounds['max_lon']) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# Add existing locations (competitors) - use red pins
print(f"Adding {len(existing_locations)} existing locations...")

for idx, row in existing_locations.iterrows():
    if row['category'] in ['supermarket', 'warehouse', 'grocery store']:
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            icon=folium.Icon(color='red', icon='shopping-cart', prefix='fa'),
            popup=f"<b>{row['name']}</b><br>Category: {row['category']}<br>Score: {row['Final_Score']:.2f}",
            tooltip=row['name']
        ).add_to(m)

# Add selected dark store locations - use blue pins ONLY
print(f"Adding {len(selected_locations)} dark store locations...")

for idx, row in selected_locations.iterrows():

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.Icon(color='blue', icon='warehouse', prefix='fa'),
        popup=f"""
        <b>Dark Store #{row['location_rank']}</b><br>
        Latitude: {row['latitude']:.6f}<br>
        Longitude: {row['longitude']:.6f}<br>
        Population Coverage: {row['population_coverage']:.2f}<br>
        Accessibility Score: {row['accessibility_score']:.4f}<br>
        Mean Delivery Time: {row['mean_delivery_time']:.2f} min<br>
        Median Delivery Time: {row['median_delivery_time']:.2f} min
        """,
        tooltip=f"Dark Store #{row['location_rank']}"
    ).add_to(m)

# Add population points (sampled) - use small gray circles
sampled_population = population_points.sample(
    min(500, len(population_points)),
    random_state=RANDOM_SEED
)

for idx, row in sampled_population.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='gray',
        fill=True,
        fillColor='gray',
        fillOpacity=0.4,
        popup=f"Population: {row['population']:.2f}",
        tooltip=f"Pop: {row['population']:.0f}"
    ).add_to(m)

# Add legend
legend_html = '''
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 280px;
    height: 150px;
    background-color: white;
    z-index:9999;
    font-size:15px;
    border:2px solid #333;
    border-radius:5px;
    padding:15px;
    box-shadow:0 0 10px rgba(0,0,0,0.2);
">
<p style="margin:0 0 10px 0; font-weight:bold; font-size:16px;">
Legend
</p>

<p style="margin:5px 0;">
<i class="fa fa-shopping-cart" style="color:red;"></i>
Existing Stores
</p>

<p style="margin:5px 0;">
<i class="fa fa-warehouse" style="color:blue;"></i>
Recommended Dark Stores
</p>

<p style="margin:5px 0;">
<i class="fa fa-circle" style="color:gray;"></i>
Population Points
</p>

</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save("dark_store_locations_map.html")

print("Interactive map saved as 'dark_store_locations_map.html'")
print("Map features:")
print(f"- {len(existing_locations)} existing stores (red pins)")
print(f"- {len(selected_locations)} dark stores (blue pins)")
print(f"- {len(sampled_population)} population points (gray circles)")
m

Adding 215 existing locations...
Adding 19 dark store locations...
Interactive map saved as 'dark_store_locations_map.html'
Map features:
- 215 existing stores (red pins)
- 19 dark stores (blue pins)
- 500 population points (gray circles)


In [ ]:
# Create comprehensive report
report = f"""
# Dark Store Location Optimization Report

## Executive Summary
This report presents the optimal locations for new dark stores in Nasr City, Cairo,
determined using a Genetic Algorithm optimization approach.

## Optimization Parameters
- Population Size: {config['population_size']}
- Number of Generations: {len(fitness_history)}
- Mutation Rate: {config['mutation_rate']}
- Crossover Rate: {config['crossover_rate']}
- Delivery Time Constraint: {config['min_delivery_time']}-{config['max_delivery_time']} minutes
- Average Delivery Speed: {config['avg_speed_kmh']} km/h

## Results
- **Total Dark Stores Recommended**: {len(selected_locations)}
- **Final Fitness Score**: {fitness_history[-1]:.4f}
- **Average Delivery Time**: {selected_locations['mean_delivery_time'].mean():.2f} minutes
- **Total Population Coverage**: {selected_locations['population_coverage'].sum():.2f}

## Recommended Dark Store Locations

| Rank | Latitude | Longitude | Population Coverage | Mean Delivery Time (min) | Median Delivery Time (min) |
|------|----------|-----------|---------------------|-------------------------|---------------------------|
"""

for idx, row in selected_locations.iterrows():
    report += f"| {row['location_rank']} | {row['latitude']:.6f} | {row['longitude']:.6f} | {row['population_coverage']:.2f} | {row['mean_delivery_time']:.2f} | {row['median_delivery_time']:.2f} |\n"

report += f"""

## Detailed Location Analysis

"""

for idx, row in selected_locations.iterrows():
    report += f"""
### Dark Store #{row['location_rank']}
- **Coordinates**: ({row['latitude']:.6f}, {row['longitude']:.6f})
- **Population Coverage**: {row['population_coverage']:.2f}
- **Accessibility Score**: {row['accessibility_score']:.4f}
- **POI Proximity Score**: {row['poi_proximity_score']:.4f}
- **Competition Penalty**: {row['competition_penalty']:.4f}
- **Delivery Time Statistics**:
  - Minimum: {row['min_delivery_time']:.2f} minutes
  - Maximum: {row['max_delivery_time']:.2f} minutes
  - Mean: {row['mean_delivery_time']:.2f} minutes
  - Median: {row['median_delivery_time']:.2f} minutes


"""

report += f"""
## Methodology

### Candidate Generation
- Generated {config['num_candidates']} candidate locations using grid-based approach
- Study area bounds: Lat [{bounds['min_lat']:.4f}, {bounds['max_lat']:.4f}], Lon [{bounds['min_lon']:.4f}, {bounds['max_lon']:.4f}]

### Feature Engineering
- Population coverage within 2km radius
- Distance to nearest road
- Distance to nearest POI
- POI count within 2km
- Distance to nearest competitor
- Competitor density within 2km

### Genetic Algorithm
- Binary chromosome representation (1 = selected, 0 = not selected)
- Tournament selection with tournament size {config['tournament_size']}
- Uniform crossover
- Bit-flip mutation
- Elitism with {config['elite_size']} elite individuals
- Multi-objective fitness function considering:
  - Population coverage (weight: {config['weight_population_coverage']})
  - Accessibility (weight: {config['weight_accessibility']})
  - POI proximity (weight: {config['weight_poi_proximity']})
  - Competition penalty (weight: {config['weight_competition']})
  - Facility count penalty (weight: {config['weight_facility_count']})

### Constraints
- Delivery time must be between {config['min_delivery_time']}-{config['max_delivery_time']} minutes
- Penalty for overlapping coverage areas
- Minimum of 1 location must be selected

## Recommendations

1. **Priority Implementation**: Start with the top-ranked locations based on population coverage.
2. **Phased Rollout**: Consider implementing locations in phases based on budget and demand.
3. **Competitive Analysis**: Monitor existing competitors in the vicinity of each selected location.
4. **Infrastructure**: Ensure adequate road infrastructure and accessibility for delivery operations.
5. **Monitoring**: Continuously monitor delivery times and adjust operations as needed.

## Conclusion

The Genetic Algorithm successfully identified {len(selected_locations)} optimal locations for new dark stores
that balance population coverage, accessibility, competition, and delivery time constraints.
The recommended locations are strategically positioned to maximize service efficiency while
minimizing competition with existing stores.

---
*Generated by Genetic Algorithm Optimization*
*Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}*
"""

# Save report
with open('dark_store_optimization_report.md', 'w', encoding='utf-8') as f:
    f.write(report)

print(report)
print("\nReport saved as 'dark_store_optimization_report.md'")


# Dark Store Location Optimization Report

## Executive Summary
This report presents the optimal locations for new dark stores in Nasr City, Cairo,
determined using a Genetic Algorithm optimization approach.

## Optimization Parameters
- Population Size: 100
- Number of Generations: 21
- Mutation Rate: 0.1
- Crossover Rate: 0.8
- Delivery Time Constraint: 6-20 minutes
- Average Delivery Speed: 30.0 km/h

## Results
- **Total Dark Stores Recommended**: 19
- **Final Fitness Score**: -39963.9934
- **Average Delivery Time**: 11.24 minutes
- **Total Population Coverage**: 5.45

## Recommended Dark Store Locations

| Rank | Latitude | Longitude | Population Coverage | Mean Delivery Time (min) | Median Delivery Time (min) |
|------|----------|-----------|---------------------|-------------------------|---------------------------|
| 1.0 | 30.015012 | 31.313749 | 0.00 | 13.19 | 13.41 |
| 2.0 | 30.015012 | 31.346390 | 0.15 | 11.67 | 11.57 |
| 3.0 | 30.020492 | 31.395351 | 0.01 | 11.34 | 11.05 |

In [ ]:
output_columns = [
    'location_rank', 'latitude', 'longitude',
    'population_coverage', 'accessibility_score', 'poi_proximity_score',
    'competition_penalty', 'min_delivery_time', 'max_delivery_time',
    'mean_delivery_time', 'median_delivery_time'
]

selected_locations[output_columns].to_csv('selected_dark_stores.csv', index=False)
print("Selected locations exported to 'selected_dark_stores.csv'")

# Export all candidates with features
candidates_normalized.to_csv('all_candidates_with_features.csv', index=False)
print("All candidates exported to 'all_candidates_with_features.csv'")

# Export fitness history
fitness_df = pd.DataFrame({'generation': range(1, len(fitness_history) + 1),
                           'best_fitness': fitness_history})
fitness_df.to_csv('ga_fitness_history.csv', index=False)
print("Fitness history exported to 'ga_fitness_history.csv'")

Selected locations exported to 'selected_dark_stores.csv'
All candidates exported to 'all_candidates_with_features.csv'
Fitness history exported to 'ga_fitness_history.csv'


In [ ]:
print("="*70)
print("DARK STORE LOCATION OPTIMIZATION - FINAL SUMMARY")
print("="*70)
print(f"\nOptimization Method: Genetic Algorithm")
print(f"Total Generations: {len(fitness_history)}")
print(f"Final Fitness Score: {fitness_history[-1]:.4f}")
print(f"\nRecommended Actions:")
print(f"- Open {len(selected_locations)} new dark stores at the specified locations")
print(f"- Expected average delivery time: {selected_locations['mean_delivery_time'].mean():.2f} minutes")
print(f"- Total population coverage: {selected_locations['population_coverage'].sum():.2f}")
print(f"\nOutput Files Generated:")
print(f"1. selected_dark_stores.csv - Recommended locations with details")
print(f"2. dark_store_optimization_report.md - Comprehensive report")
print(f"3. dark_store_locations_map.html - Interactive map")
print(f"4. ga_convergence.png - Optimization convergence plot")
print(f"5. all_candidates_with_features.csv - All candidate locations")
print(f"6. ga_fitness_history.csv - Fitness evolution over generations")
print("\n" + "="*70)
print("Optimization completed successfully!")
print("="*70)

DARK STORE LOCATION OPTIMIZATION - FINAL SUMMARY

Optimization Method: Genetic Algorithm
Total Generations: 21
Final Fitness Score: -39963.9934

Recommended Actions:
- Open 19 new dark stores at the specified locations
- Expected average delivery time: 11.24 minutes
- Total population coverage: 5.45

Output Files Generated:
1. selected_dark_stores.csv - Recommended locations with details
2. dark_store_optimization_report.md - Comprehensive report
3. dark_store_locations_map.html - Interactive map
4. ga_convergence.png - Optimization convergence plot
5. all_candidates_with_features.csv - All candidate locations
6. ga_fitness_history.csv - Fitness evolution over generations

Optimization completed successfully!
